<a href="https://colab.research.google.com/github/alfnihilus/computational-chem/blob/main/self-study/creat_molecul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# First, install python library
!pip install pubchempy rdkit pyopsin ase

In [ ]:
# @markdown # Also Run This

!java -version
import requests
import os

def download_opsin_latest():
    if os.path.exists("opsin.jar"):
        print("OPSIN sudah ada, skip download.")
        return

    # Ambil info versi terbaru dari GitHub API
    api_url = "https://api.github.com/repos/dan2097/opsin/releases/latest"
    response = requests.get(api_url)
    data = response.json()

    latest_version = data["tag_name"]
    print(f"Versi OPSIN terbaru: {latest_version}")

    # Cari file JAR di assets
    jar_url = None
    for asset in data["assets"]:
        if asset["name"].endswith("jar-with-dependencies.jar"):
            jar_url = asset["browser_download_url"]
            break

    if not jar_url:
        print("JAR tidak ditemukan di release terbaru!")
        return

    # Download JAR
    print(f"Mendownload dari: {jar_url}")
    jar_data = requests.get(jar_url)
    with open("opsin.jar", "wb") as f:
        f.write(jar_data.content)
    print("Download selesai!")

download_opsin_latest()

In [ ]:
import pubchempy as pcp
import subprocess
import os

# @markdown # Pembuatan SMILES
# @markdown ### Masukkan Nama IUPAC (von Baeyer Style) atau Nama Umum/Trivial (Cth. Etanol, 2,4,6-trinitrotoluene)
input_nama_senyawa = ""  # @param {type:"string"}

def opsin_name_to_smiles(name, opsin_jar_path="opsin.jar"):
    """Convert IUPAC name to SMILES using OPSIN JAR."""
    if not os.path.exists(opsin_jar_path):
        print(f"OPSIN JAR not found at '{opsin_jar_path}'. Download from: https://github.com/dan2097/opsin/releases")
        return None
    try:
        result = subprocess.run(
            ["java", "-jar", opsin_jar_path],
            input=name,
            capture_output=True,
            text=True,
            timeout=30
        )
        smiles = result.stdout.strip()
        return smiles if smiles else None
    except Exception as e:
        print(f"Error running OPSIN: {e}")
        return None

def cek_smiles_lengkap(nama_atau_rumus):
    # Coba cari di PubChem terlebih dahulu
    results = pcp.get_compounds(nama_atau_rumus, 'name')

    if results:
        print(f"Ditemukan di PubChem untuk: {nama_atau_rumus}")
        for compound in results:
            print(f"Nama         : {compound.iupac_name}")
            print(f"Canonical SMILES : {compound.canonical_smiles}")
            print(f"Isomeric SMILES  : {compound.isomeric_smiles}")
            print("-" * 30)
        return

    # Fallback ke OPSIN
    print(f"Tidak ditemukan di PubChem untuk '{nama_atau_rumus}', mencoba membaca via OPSIN...")
    smiles_opsin = opsin_name_to_smiles(nama_atau_rumus, opsin_jar_path="opsin.jar")

    if smiles_opsin:
        print(f"Ditemukan oleh OPSIN untuk: {nama_atau_rumus}")
        print(f"SMILES dari OPSIN: {smiles_opsin}")
    else:
        print(f"Gagal: Nama '{nama_atau_rumus}' tidak dikenali oleh PubChem maupun OPSIN.")

# Memanggil fungsi
cek_smiles_lengkap(input_nama_senyawa)

In [ ]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem
from ase import Atoms

def generate_molecule(identifier):
    try:
        results = []
        search_type = ""

        # 1. Coba cari berdasarkan CID jika identifier adalah angka murni
        if identifier.isdigit():
            print(f"Mencari sebagai CID: {identifier}...")
            results = pcp.get_compounds(identifier, 'cid')
            if results:
                search_type = "CID"

        # 2. Jika belum ditemukan, coba cari sebagai SMILES
        #    SMILES seringkali mengandung '=', '#', '(', ')', '[', ']', '.', '@', '/', '\\'
        #    dan tidak boleh hanya angka. Periksa beberapa karakter khas SMILES.
        if not results and (any(c in identifier for c in ['=', '#', '(', ')', '[', ']', '.', '@', '/', '\\']) or (not identifier.isalpha() and not identifier.isdigit())):
            print(f"Tidak ditemukan sebagai CID. Mencoba sebagai SMILES: {identifier}...")
            results = pcp.get_compounds(identifier, 'smiles')
            if results:
                search_type = "SMILES"

        # 3. Jika belum ditemukan, coba cari sebagai Nama
        if not results:
            print(f"Tidak ditemukan sebagai CID/SMILES. Mencoba sebagai Nama: {identifier}...")
            results = pcp.get_compounds(identifier, 'name')
            if results:
                search_type = "Nama"

        # 4. Jika belum ditemukan, coba cari berdasarkan Rumus
        if not results:
            print(f"Tidak ditemukan sebagai CID/SMILES/Nama. Mencoba sebagai Rumus: {identifier}...")
            results = pcp.get_compounds(identifier, 'formula')
            if results:
                search_type = "Rumus"

        if not results:
            print("Molekul tidak ditemukan di database PubChem.")
            return

        # Mengambil hasil pertama yang ditemukan
        comp = results[0]
        smiles = comp.smiles
        print(f"Ditemukan! Tipe: {search_type}, SMILES: {smiles}")

        # 3. Proses RDKit untuk Koordinat 3D
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Error: Tidak dapat mengonversi SMILES '{smiles}' menjadi objek molekul RDKit.")
            return

        mol = Chem.AddHs(mol)

        # Memberikan koordinat 3D awal
        # Menggunakan AllChem.ETKDGv2() untuk hasil yang lebih baik
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv2())
        # Optimasi struktur agar stabil (Universal Force Field)
        AllChem.UFFOptimizeMolecule(mol)

        # 4. Cetak koordinat
        print(f"\nKoordinat untuk {identifier} (dari {search_type}):")
        conf = mol.GetConformer()
        for i, atom in enumerate(mol.GetAtoms()):
            pos = conf.GetAtomPosition(i)
            print(f"{atom.GetSymbol()} {pos.x:.4f} {pos.y:.4f} {pos.z:.4f}")

    except Exception as e:
        print(f"Error: {e}")

# Input dari user
# @markdown # Pembuatan Koordinat Secara Otomatis (Sumber PubChem)
# @markdown ### Masukkan Nama atau Rumus Kimia (misal: H2O, Ethanol, C6H6, Flavonoid, 2244, CCCCCCCCCCCC)
user_input ="" # @param {type:"string"}
generate_molecule(user_input)
